# Quran Multilingual Corpus — Tokenizer Fertility Experiment

**Meaning-controlled tokenizer fairness audit.** All 40+ translations render the *same* 6,236 verses, so differences in tokens-per-verse isolate the language/script/tokenizer effect from content.

Steps: load CSVs + Arabic source -> verify alignment -> measure fertility with real production tokenizers -> English-vs-multilingual BPE contrast -> cost simulation -> save & zip results.

**Runtime:** CPU is fine (tokenizers only; no model weights loaded).

## 0. Install dependencies

In [ ]:
!pip -q install tiktoken transformers sentencepiece tokenizers pandas matplotlib 2>/dev/null
import os, io, csv, json, glob, time, zipfile, statistics
import pandas as pd, numpy as np
print('deps ready')

## 1. Provide the data
Upload `translations.zip` (the 41 CSVs) when prompted. Or mount Drive and set `DATA_DIR` (Option B).

In [ ]:
# ---- Option A: upload a zip of the CSVs ----
from google.colab import files
DATA_DIR = '/content/translations'
os.makedirs(DATA_DIR, exist_ok=True)
print('Upload translations.zip:')
up = files.upload()
for fn in up:
    if fn.lower().endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(up[fn])) as z: z.extractall(DATA_DIR)
        print('extracted', fn)
    elif fn.lower().endswith('.csv'):
        open(os.path.join(DATA_DIR, fn),'wb').write(up[fn])

# ---- Option B: Google Drive (uncomment) ----
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/Download Translations'

csv_files = sorted(glob.glob(os.path.join(DATA_DIR,'**','*.csv'), recursive=True))
print(len(csv_files),'CSV files found')
for f in csv_files[:5]: print('  ', os.path.basename(f))

## 2. Parse translations
Skip the comment preamble, then key every verse by `(sura, aya)`.

In [ ]:
def load_csv(fp):
    keys={}
    with open(fp, encoding='utf-8', errors='replace') as f:
        rdr=csv.reader(f); started=False; iS=iA=iT=None
        for row in rdr:
            if not started:
                low=[(c or '').strip().lower() for c in row]
                if 'sura' in low and 'aya' in low:
                    started=True; iS=low.index('sura'); iA=low.index('aya')
                    iT=low.index('translation') if 'translation' in low else 3
                continue
            try:
                keys[(int(row[iS]), int(row[iA]))]=row[iT]
            except Exception:
                continue
    return keys

def lang_name(fp): return os.path.basename(fp).split('_')[0]

data={fp: load_csv(fp) for fp in csv_files}
from collections import Counter
cnt=Counter(lang_name(f) for f in csv_files)
disp={f:(lang_name(f) if cnt[lang_name(f)]==1 else lang_name(f)+'-'+os.path.basename(f).split('_')[1]) for f in csv_files}
print('Loaded', len(data), 'translations')
print('Verse 1:1 sample:', next(iter(data.values()))[(1,1)][:80])

## 3. Fetch the Arabic source (Uthmani / Hafs) — fertility baseline

In [ ]:
import urllib.request
ARABIC_URL='https://raw.githubusercontent.com/risan/quran-json/main/dist/quran.json'
arabic={}
try:
    raw=urllib.request.urlopen(ARABIC_URL, timeout=60).read().decode('utf-8')
    for sura in json.loads(raw):
        for v in sura['verses']:
            arabic[(sura['id'], v['id'])]=v['text']
    print('Arabic verses:', len(arabic))
    data['arabic_source']=arabic
    disp['arabic_source']='arabic_source'
except Exception as e:
    print('Arabic fetch failed:', e)

## 4. Verify alignment -> `alignment_report.csv`

In [ ]:
allkeys=set()
for k in data.values(): allkeys|=set(k.keys())
common=None
for k in data.values():
    common = set(k.keys()) if common is None else (common & set(k.keys()))
common=sorted(common)
print('Union:', len(allkeys), '| Common across ALL:', len(common), '| Expected (Hafs): 6236')
rows=[{'file':disp[fp],'verses':len(kd),'missing_vs_union':len(allkeys-set(kd)),
       'empty_cells':sum(1 for v in kd.values() if not str(v).strip())} for fp,kd in data.items()]
align=pd.DataFrame(rows).sort_values('file')
align.to_csv('alignment_report.csv', index=False)
print('saved alignment_report.csv'); align.head(10)

## 5. Production tokenizers

`tiktoken` = OpenAI (no auth). HuggingFace = the rest. Each gated model tries its official ID first, then falls back to an **ungated mirror** so Llama-3 / Gemma / Aya load without any login. Everything is wrapped in try/except; failures are skipped and the run still completes.

### 5a. (Optional) HuggingFace login for the *official* gated repos
Only needed if you want the official `meta-llama/Meta-Llama-3-8B`, `google/gemma-7b`, `CohereForAI/aya-23-8B` instead of the ungated mirrors. Accept each license on huggingface.co first, then uncomment and run.

In [ ]:
# from huggingface_hub import login
# login('hf_PASTE_YOUR_READ_TOKEN')
print('Login is optional — ungated mirrors work without it.')

### 5b. Load tokenizers

In [ ]:
import tiktoken
from transformers import AutoTokenizer

def make_tiktoken(name):
    enc=tiktoken.get_encoding(name)
    return lambda texts: [len(t) for t in enc.encode_batch(texts)]

def make_hf(model_id):
    try:
        tok=AutoTokenizer.from_pretrained(model_id)
    except Exception:
        tok=AutoTokenizer.from_pretrained(model_id, use_fast=False)
    def f(texts):
        return [len(ids) for ids in tok(texts, add_special_tokens=False)['input_ids']]
    return f

TOKENIZERS={}
# (name, kind, primary_ref, [fallback_refs...])
specs=[
    ('gpt4o_o200k', 'tiktoken', 'o200k_base', []),
    ('gpt4_cl100k', 'tiktoken', 'cl100k_base', []),
    ('mbert',  'hf', 'bert-base-multilingual-cased', []),
    ('xlmr',   'hf', 'xlm-roberta-base', []),
    ('bloom',  'hf', 'bigscience/bloom-560m', []),
    ('byt5',   'hf', 'google/byt5-small', []),
    ('nllb',   'hf', 'facebook/nllb-200-distilled-600M', []),
    ('mt5',    'hf', 'google/mt5-small', []),
    ('llama3', 'hf', 'meta-llama/Meta-Llama-3-8B', ['NousResearch/Meta-Llama-3-8B', 'unsloth/llama-3-8b']),
    ('gemma',  'hf', 'google/gemma-7b', ['unsloth/gemma-2-2b', 'unsloth/gemma-2b']),
    ('aya',    'hf', 'CohereForAI/aya-23-8B', ['CohereForAI/aya-101']),
]
for name, kind, primary, fallbacks in specs:
    loaded=False
    for ref in [primary]+fallbacks:
        try:
            TOKENIZERS[name]= make_tiktoken(ref) if kind=='tiktoken' else make_hf(ref)
            print('loaded', name, '<-', ref, '(mirror)' if ref!=primary else '')
            loaded=True; break
        except Exception:
            continue
    if not loaded:
        print('skipped', name, '(all sources failed)')
print('\nActive tokenizers:', list(TOKENIZERS))

## 6. Compute production fertility -> `fertility_production.csv`

In [ ]:
keys=common; nv=len(keys)
prod_rows=[]
for fp in data:
    texts=[str(data[fp][k]) for k in keys]
    row={'language':disp[fp],
         'bytes_per_verse': sum(len(t.encode('utf-8')) for t in texts)/nv,
         'chars_per_verse': sum(len(t) for t in texts)/nv,
         'words_per_verse': sum(len(t.split()) for t in texts)/nv}
    for name,fn in TOKENIZERS.items():
        try: row[name+'_tok_per_verse']=sum(fn(texts))/nv
        except Exception: row[name+'_tok_per_verse']=np.nan
    prod_rows.append(row); print('done', disp[fp])
prod=pd.DataFrame(prod_rows)
tok_cols=[c for c in prod.columns if c.endswith('_tok_per_verse')]
def base(lbl):
    m=prod[prod.language==lbl]; return m.iloc[0] if len(m) else None
eng=base('english'); ar=base('arabic_source')
for col in tok_cols:
    if eng is not None and pd.notna(eng[col]) and eng[col]>0:
        prod[col.replace('_tok_per_verse','_ratio_vs_en')]=prod[col]/eng[col]
    if ar is not None and pd.notna(ar[col]) and ar[col]>0:
        prod[col.replace('_tok_per_verse','_ratio_vs_ar')]=prod[col]/ar[col]
prod=prod.sort_values(tok_cols[0], ascending=False)
prod.to_csv('fertility_production.csv', index=False)
print('saved fertility_production.csv', prod.shape)
prod[['language']+tok_cols].head(12)

## 7. Controlled contrast: English-trained vs multilingual-trained BPE
Same 32k vocab budget; only the training mix differs. Isolates vocabulary composition as the cause of the gap. -> `fertility_controlled.csv`

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
VOCAB=32000
def train_bpe(corpus):
    t=Tokenizer(models.BPE(unk_token='[UNK]'))
    t.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False)
    t.train_from_iterator(corpus, trainer=trainers.BpeTrainer(vocab_size=VOCAB, special_tokens=['[UNK]'], show_progress=False))
    return t
eng_fp=[f for f in data if disp[f]=='english'][0]
tok_en=train_bpe([str(data[eng_fp][k]) for k in keys])
tok_mu=train_bpe([str(data[f][k]) for f in data for k in keys])
def fert(tk,texts): return sum(len(e.ids) for e in tk.encode_batch(texts))/nv
ctrl_rows=[]
for fp in data:
    texts=[str(data[fp][k]) for k in keys]
    ctrl_rows.append({'language':disp[fp],
                      'en_bpe_tok_per_verse':fert(tok_en,texts),
                      'multi_bpe_tok_per_verse':fert(tok_mu,texts),
                      'bytes_per_verse':sum(len(t.encode('utf-8')) for t in texts)/nv})
ctrl=pd.DataFrame(ctrl_rows)
en_b=ctrl[ctrl.language=='english']['en_bpe_tok_per_verse'].iloc[0]
ctrl['ratio_vs_english']=ctrl['en_bpe_tok_per_verse']/en_b
if 'arabic_source' in ctrl.language.values:
    ar_b=ctrl[ctrl.language=='arabic_source']['en_bpe_tok_per_verse'].iloc[0]
    ctrl['ratio_vs_arabic']=ctrl['en_bpe_tok_per_verse']/ar_b
ctrl=ctrl.sort_values('en_bpe_tok_per_verse', ascending=False)
ctrl.to_csv('fertility_controlled.csv', index=False)
sp_en=ctrl['en_bpe_tok_per_verse']; sp_mu=ctrl['multi_bpe_tok_per_verse']
print('EN-BPE spread max/min = %.1fx'%(sp_en.max()/sp_en.min()))
print('MULTI-BPE spread max/min = %.1fx'%(sp_mu.max()/sp_mu.min()))
print('saved fertility_controlled.csv'); ctrl.head(12)

## 8. Cost / context-window simulation -> `cost_simulation.csv`

In [ ]:
PRICE_PER_M=5.00   # $ per 1M input tokens (edit to current price)
CONTEXT=128000
PRIMARY=tok_cols[0]
sim=prod[['language',PRIMARY]].copy()
sim['usd_per_full_quran']=sim[PRIMARY]*nv/1e6*PRICE_PER_M
sim['verses_per_context']=(CONTEXT/sim[PRIMARY]).round(0)
sim=sim.sort_values(PRIMARY, ascending=False)
sim.to_csv('cost_simulation.csv', index=False)
print('saved cost_simulation.csv'); sim.head(12)

## 9. Chart -> `fertility_chart.png`

In [ ]:
import matplotlib.pyplot as plt
d=ctrl[ctrl.language!='arabic_source'].sort_values('en_bpe_tok_per_verse')
y=np.arange(len(d))
fig,ax=plt.subplots(figsize=(10,12))
ax.barh(y+0.2, d['en_bpe_tok_per_verse'], height=0.4, color='#e76f51', label='English-trained 32k BPE')
ax.barh(y-0.2, d['multi_bpe_tok_per_verse'], height=0.4, color='#2a9d8f', label='Multilingual 32k BPE')
ax.set_yticks(y); ax.set_yticklabels(d['language'], fontsize=8)
ax.set_xlabel('Tokens per verse (identical meaning across languages)')
ax.set_title('Tokenizer fertility - meaning-controlled Quran corpus')
ax.legend(loc='lower right'); ax.grid(axis='x', alpha=.25)
plt.tight_layout(); plt.savefig('fertility_chart.png', dpi=140)
print('saved fertility_chart.png')

## 10. Bundle results -> `quran_fertility_results.zip` (download & upload back)

In [ ]:
out=[f for f in ['alignment_report.csv','fertility_production.csv','fertility_controlled.csv','cost_simulation.csv','fertility_chart.png'] if os.path.exists(f)]
json.dump({'n_languages':len(data),'verses':nv,'tokenizers':list(TOKENIZERS),'vocab_size_bpe':VOCAB,
           'price_per_M':PRICE_PER_M,'context':CONTEXT,'timestamp':time.strftime('%Y-%m-%d %H:%M')},
          open('run_metadata.json','w'), indent=2)
out.append('run_metadata.json')
with zipfile.ZipFile('quran_fertility_results.zip','w') as z:
    for f in out: z.write(f)
print('zipped:', out)
from google.colab import files; files.download('quran_fertility_results.zip')